# HW3 — Two retrievers and a model: what RAG actually sends

**Name:**
**Student ID:**
**Group:**
**Repository:** https://github.com/Yoroto-crypto/css4007-hw3

## AI tool disclosure

State which AI tools you used and for what. Expected and fine; undisclosed use is not.

> I used Claude (Anthropic's AI assistant) during this homework: to explain the terminology (RAG, BM25, stemming, cosine similarity, tokens), to work through and check the Part A calculations, and to draft the code for Parts B and C and the answers to the conceptual questions.

Run this notebook **top to bottom** before you push. A cell with no saved output earns nothing.
The instructions for every section are in `README.md`.

In [6]:
%pip install -r requirements.txt

  Using cached tiktoken-0.14.0-cp312-cp312-win_amd64.whl.metadata (6.8 kB)
  Using cached anyio-4.15.1-py3-none-any.whl.metadata (4.7 kB)
  Using cached jiter-0.17.0-cp312-cp312-win_amd64.whl.metadata (5.3 kB)
  Using cached typing_extensions-4.16.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached idna-3.20-py3-none-any.whl.metadata (7.2 kB)
  Using cached truststore-0.10.4-py3-none-any.whl.metadata (4.4 kB)
INFO: pip is looking at multiple versions of httpcore2 to determine which version is compatible with other requirements. This could take a while.
  Using cached httpx2-2.13.0-py3-none-any.whl.metadata (10 kB)
  Using cached httpcore2-2.13.0-py3-none-any.whl.metadata (26 kB)
  Using cached httpx2-2.12.0-py3-none-any.whl.metadata (9.5 kB)
  Using cached httpcore2-2.12.0-py3-none-any.whl.metadata (25 kB)
  Using cached h11-0.16.0-py3-none-any.whl.metadata (8.3 kB)
  Using cached isoduration-20.11.0-py3-none-any.whl.metadata (5.7 kB)
  Using cached uri_template-1.3.0-py3-none-any.whl.

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
conda 25.1.1 requires conda-libmamba-solver>=24.11.0, but you have conda-libmamba-solver 24.9.0 which is incompatible.


---
# Part A — relevance by hand (1 pt)

| | Text |
|---|---|
| D1 | Large language models can generate text and answer questions. |
| D2 | Retrieval-Augmented Generation retrieves relevant documents before generating an answer. |
| D3 | Embeddings represent text as numerical vectors. |
| Query | How does RAG retrieve information? |

`k1 = 1.5`, `b = 0.75`, `idf(t) = ln((N − n(t) + 0.5) / (n(t) + 0.5))`. Tokenise as README §3 says.

## A1 (a) — BM25 as written

**Tokens of each document, and its length:**

| Doc | Tokens | \|d\| |
|---|---|---:|
| D1 | large, language, models, can, generate, text, and, answer, questions | 9 |
| D2 | retrieval, augmented, generation, retrieves, relevant, documents, before, generating, an, answer | 10 |
| D3 | embeddings, represent, text, as, numerical, vectors | 6 |

**avgdl =** (9 + 10 + 6) / 3 = 25 / 3 ≈ 8.333

**Query terms:** how, does, rag, retrieve, information (N = 3)

**TF, DF and IDF of each query term:**

| Query term | f(t, D1) | f(t, D2) | f(t, D3) | n(t) (DF) | idf(t) |
|---|---:|---:|---:|---:|---:|
| how | 0 | 0 | 0 | 0 | ln(3.5 / 0.5) = ln 7 ≈ 1.946 |
| does | 0 | 0 | 0 | 0 | 1.946 |
| rag | 0 | 0 | 0 | 0 | 1.946 |
| retrieve | 0 | 0 | 0 | 0 | 1.946 |
| information | 0 | 0 | 0 | 0 | 1.946 |

(D2 contains `retrieval` and `retrieves`, but not the exact token `retrieve`, so f = 0.)

**BM25 of each document** (write out the sum for every document, one line per term that contributes):

> Every query term has f(t, d) = 0 in every document, so every term contributes
> idf · 0 · (k1 + 1) / (0 + k1 · (1 − b + b · |d| / avgdl)) = 0.
>
> - D1 = 0 (how) + 0 (does) + 0 (rag) + 0 (retrieve) + 0 (information) = **0**
> - D2 = 0 + 0 + 0 + 0 + 0 = **0**
> - D3 = 0 + 0 + 0 + 0 + 0 = **0**
>
> No term contributes anywhere. The large idf of the absent terms (1.946) does not matter, because it is multiplied by f = 0.

| Document | BM25 Score | Rank |
|---|---:|---:|
| D1 | 0 | 1 (tie) |
| D2 | 0 | 1 (tie) |
| D3 | 0 | 1 (tie) |

## A1 (b) — BM25 with the stems from README §3

**Stemmed tokens of each document, and its length:**

| Doc | Tokens | \|d\| |
|---|---|---:|
| D1 | large, language, models, can, **gener**, text, and, answer, questions | 9 |
| D2 | **retriev**, augmented, **gener**, **retriev**, relevant, documents, before, **gener**, an, answer | 10 |
| D3 | embeddings, represent, text, as, numerical, vectors | 6 |

Stemming replaces words one for one, so the lengths do not change.

**avgdl =** 25 / 3 ≈ 8.333

**Stemmed query terms:** how, doe, rag, retriev, inform

| Query term | f(t, D1) | f(t, D2) | f(t, D3) | n(t) (DF) | idf(t) |
|---|---:|---:|---:|---:|---:|
| how | 0 | 0 | 0 | 0 | ln(3.5 / 0.5) ≈ 1.946 |
| doe | 0 | 0 | 0 | 0 | 1.946 |
| rag | 0 | 0 | 0 | 0 | 1.946 |
| retriev | 0 | 2 | 0 | 1 | ln((3 − 1 + 0.5) / (1 + 0.5)) = ln(5/3) ≈ 0.5108 |
| inform | 0 | 0 | 0 | 0 | 1.946 |

**BM25 of each document**, worked:

> Only `retriev` has f > 0, and only in D2.
>
> For D2: |d| / avgdl = 10 / 8.333 = 1.2, so 1 − b + b · 1.2 = 0.25 + 0.75 · 1.2 = 1.15.
> k1 · 1.15 = 1.725.
>
> term(retriev, D2) = 0.5108 · 2 · (1.5 + 1) / (2 + 1.725) = 0.5108 · 5 / 3.725 = 2.554 / 3.725 ≈ **0.6857**
>
> The other four terms have f = 0 and contribute 0.
>
> - D1 = 0 + 0 + 0 + 0 + 0 = **0**
> - D2 = 0 + 0 + 0 + 0.6857 + 0 = **0.6857**
> - D3 = 0 + 0 + 0 + 0 + 0 = **0**

| Document | BM25 Score | Rank |
|---|---:|---:|
| D1 | 0 | 2 (tie) |
| D2 | 0.6857 | 1 |
| D3 | 0 | 2 (tie) |

**Which document does BM25 retrieve first, in (a) and in (b)? Why?**

> **(a)** None: all three scores are 0, a three-way tie. This is not an arithmetic mistake, I rechecked it. BM25 only matches exact tokens, and the query token `retrieve` appears in no document (D2 has `retrieves` and `retrieval`). The other query words (`how`, `does`, `rag`, `information`) are not in any document either. So D2, the obviously right answer, scores 0 like the rest.
>
> **(b)** D2, with 0.6857. After stemming, `retrieves` and `retrieval` in D2 and `retrieve` in the query all become `retriev`, so f(retriev, D2) = 2 and D2 is the only document with a matching term. D1 and D3 stay at 0. Note that stemming still cannot connect `rag` to "Retrieval-Augmented Generation" or `information` to anything in the documents: the only match comes from one shared word root.

## A2 — cosine similarity

```
Query = [0.8, 0.2, 0.5]
D1 = [0.7, 0.1, 0.4]   D2 = [0.9, 0.3, 0.6]   D3 = [0.1, 0.8, 0.2]
```

| Pair | Dot product (worked) | ‖Query‖ | ‖D‖ | Cosine |
|---|---|---:|---:|---:|
| Query · D1 | 0.8·0.7 + 0.2·0.1 + 0.5·0.4 = 0.56 + 0.02 + 0.20 = 0.78 | √0.93 ≈ 0.9644 | √0.66 ≈ 0.8124 | 0.78 / (0.9644 · 0.8124) ≈ 0.9956 |
| Query · D2 | 0.8·0.9 + 0.2·0.3 + 0.5·0.6 = 0.72 + 0.06 + 0.30 = 1.08 | 0.9644 | √1.26 ≈ 1.1225 | 1.08 / (0.9644 · 1.1225) ≈ 0.9977 |
| Query · D3 | 0.8·0.1 + 0.2·0.8 + 0.5·0.2 = 0.08 + 0.16 + 0.10 = 0.34 | 0.9644 | √0.69 ≈ 0.8307 | 0.34 / (0.9644 · 0.8307) ≈ 0.4244 |

| Document | Cosine Similarity | Rank |
|---|---:|---:|
| D1 | 0.9956 | 2 |
| D2 | 0.9977 | 1 |
| D3 | 0.4244 | 3 |

**Which document is most similar to the query?**

> D2 (cosine ≈ 0.9977). D1 is almost as close (≈ 0.9956): both vectors point in nearly the same direction as the query. D3 is far away (≈ 0.4244).

**Is the ranking the same as BM25's?** Compare it with both A1 rankings.

> Cosine ranks D2 > D1 > D3.
> - **BM25 (a):** no ranking at all (D1 = D2 = D3 = 0), so it disagrees: it cannot even pick D2.
> - **BM25 (b):** D2 first, then a tie between D1 and D3 (both 0). The top-1 agrees with cosine, but BM25 cannot tell D1 from D3, while cosine puts D1 clearly above D3 (0.996 vs 0.424).

**Why can BM25 and embedding retrieval disagree?** Point at the words of this query and these documents.

> BM25 needs the same token on both sides. The query says `retrieve`, D2 says `retrieves` and `retrieval`: without stemming there is no match. The query says `RAG`, D2 spells it out as "Retrieval-Augmented Generation": BM25 cannot know the abbreviation and the phrase mean the same thing. `information` appears in no document.
>
> An embedding model compares meaning, not spelling. It can place "RAG" near "Retrieval-Augmented Generation", and "retrieve information" near D2's "retrieves relevant documents" and D1's "answer questions", so D2 and D1 get high cosines. D3 (about vectors and numerical representation of text) is about a different thing and gets a low one. So BM25 fails when the words differ but the meaning is the same, and embeddings handle that case.

Optional: check your Part A arithmetic here. The working above must still be written out by hand.

In [1]:
import math

k1, b = 1.5, 0.75

def bm25(docs, query):
    N = len(docs)
    avgdl = sum(len(d) for d in docs) / N
    print("lengths:", [len(d) for d in docs], "avgdl =", round(avgdl, 4))
    scores = [0.0] * N
    for t in query:
        n = sum(t in d for d in docs)
        idf = math.log((N - n + 0.5) / (n + 0.5))
        print(f"  {t:12s} f={[d.count(t) for d in docs]} n={n} idf={idf:.4f}")
        for i, d in enumerate(docs):
            f = d.count(t)
            scores[i] += idf * f * (k1 + 1) / (f + k1 * (1 - b + b * len(d) / avgdl))
    return [round(s, 4) for s in scores]

raw = [
    "large language models can generate text and answer questions".split(),
    "retrieval augmented generation retrieves relevant documents before generating an answer".split(),
    "embeddings represent text as numerical vectors".split(),
]
print("(a) as written")
print("  scores D1, D2, D3:", bm25(raw, "how does rag retrieve information".split()))

stem = {"retrieve": "retriev", "retrieves": "retriev", "retrieval": "retriev",
        "generate": "gener", "generating": "gener", "generation": "gener",
        "information": "inform", "does": "doe"}
stemmed = [[stem.get(w, w) for w in d] for d in raw]
print("(b) with stemming")
print("  scores D1, D2, D3:", bm25(stemmed, [stem.get(w, w) for w in "how does rag retrieve information".split()]))

print("A2 cosine")
Q = [0.8, 0.2, 0.5]
for name, D in [("D1", [0.7, 0.1, 0.4]), ("D2", [0.9, 0.3, 0.6]), ("D3", [0.1, 0.8, 0.2])]:
    dot = sum(x * y for x, y in zip(Q, D))
    nq = math.sqrt(sum(x * x for x in Q)); nd = math.sqrt(sum(x * x for x in D))
    print(f"  {name}: dot={dot:.2f} |Q|={nq:.4f} |D|={nd:.4f} cos={dot / (nq * nd):.4f}")


(a) as written
lengths: [9, 10, 6] avgdl = 8.3333
  how          f=[0, 0, 0] n=0 idf=1.9459
  does         f=[0, 0, 0] n=0 idf=1.9459
  rag          f=[0, 0, 0] n=0 idf=1.9459
  retrieve     f=[0, 0, 0] n=0 idf=1.9459
  information  f=[0, 0, 0] n=0 idf=1.9459
  scores D1, D2, D3: [0.0, 0.0, 0.0]
(b) with stemming
lengths: [9, 10, 6] avgdl = 8.3333
  how          f=[0, 0, 0] n=0 idf=1.9459
  doe          f=[0, 0, 0] n=0 idf=1.9459
  rag          f=[0, 0, 0] n=0 idf=1.9459
  retriev      f=[0, 2, 0] n=1 idf=0.5108
  inform       f=[0, 0, 0] n=0 idf=1.9459
  scores D1, D2, D3: [0.0, 0.6857, 0.0]
A2 cosine
  D1: dot=0.78 |Q|=0.9644 |D|=0.8124 cos=0.9956
  D2: dot=1.08 |Q|=0.9644 |D|=1.1225 cos=0.9977
  D3: dot=0.34 |Q|=0.9644 |D|=0.8307 cos=0.4244


---
# Part B — the same two retrievers, from libraries (1.5 pt)

Load `data/corpus.jsonl` and `data/queries.json`.

In [2]:
import json, os, re
import numpy as np
from dotenv import load_dotenv

load_dotenv()   # reads OPENAI_API_KEY and HF_TOKEN from .env (never print them)

with open("data/corpus.jsonl", encoding="utf-8") as f:
    corpus = [json.loads(line) for line in f]
ids = [c["id"] for c in corpus]
texts = [c["text"] for c in corpus]

with open("data/queries.json", encoding="utf-8") as f:
    fixed_queries = json.load(f)["queries"]

print(len(corpus), "chunks,", len(fixed_queries), "fixed queries")


74 chunks, 8 fixed queries


## B1 — BM25 with `rank_bm25`

**My tokenisation:** (lowercase? regex? stemming? stop words?)

> Lowercase, then `re.findall(r"\w+", text)`: runs of letters and digits, punctuation dropped. `\w` is Unicode-aware, so Kazakh and Russian words stay whole. No stemming, no stop-word removal. Because `\w+` splits at hyphens, `GR-305` becomes two tokens, `gr` and `305`: the number is the part that tells the forms apart.

In [7]:
from rank_bm25 import BM25Okapi

def tokenize(s):
    return re.findall(r"\w+", s.lower())

bm25 = BM25Okapi([tokenize(t) for t in texts], k1=1.5, b=0.75)

def first_line(i, n=70):
    return texts[i].split("\n")[0][:n]

def show(label, order, scores):
    print(" ", label)
    for rank, i in enumerate(order, 1):
        print(f"    {rank}. {ids[i]}  {scores[i]:.4f}  {first_line(i)}")

# quick demonstration on one fixed query
q = fixed_queries[0]
s = bm25.get_scores(tokenize(q["text"]))
print(q["id"], q["text"])
show("BM25", np.argsort(-s)[:3], s)


Q-01 What is form GR-305 used for?
  BM25
    1. GO-08  15.8255  Form GR-305 is used to appeal a decision of the Grant Office. It is th
    2. GO-12  8.1953  A student who disagrees with a decision may appeal within 14 calendar 
    3. GO-06  8.0700  Form GR-104 is the application form for the need-based study grant. It


## B2 — EmbeddingGemma (`google/embeddinggemma-300m`) with `encode_document` / `encode_query`

In [13]:
import os
from dotenv import load_dotenv, find_dotenv

print("найден .env:", find_dotenv(usecwd=True))
load_dotenv(find_dotenv(usecwd=True), override=True)

t = os.getenv("HF_TOKEN")
print("токен загружен:", bool(t), "| начинается с hf_:", bool(t) and t.startswith("hf_"))

найден .env: c:\Users\ASUS\Desktop\4 курс 1 семестр\Искусственный интеллект\css4007-hw3-main\.env
токен загружен: True | начинается с hf_: True


In [10]:
import os
print("рабочая папка:", os.getcwd())
print([f for f in os.listdir(".") if "env" in f.lower()])

рабочая папка: c:\Users\ASUS\Desktop\4 курс 1 семестр\Искусственный интеллект\css4007-hw3-main
['.env.example']


In [11]:
copy .env.example .env

�����஢��� 䠩���:         1.


In [16]:
from sentence_transformers import SentenceTransformer

# Colab: put HF_TOKEN in the Secrets panel and use  from google.colab import userdata
model = SentenceTransformer("google/embeddinggemma-300m", token=os.getenv("HF_TOKEN"))

doc_emb = model.encode_document(texts)       # float32, one vector per chunk
print("document embeddings:", doc_emb.shape)

def emb_scores(query):
    q_emb = model.encode_query(query)
    return np.asarray(model.similarity(q_emb, doc_emb))[0]   # cosine similarity to every chunk

s = emb_scores(fixed_queries[0]["text"])
print(fixed_queries[0]["id"], fixed_queries[0]["text"])
show("EmbeddingGemma", np.argsort(-s)[:3], s)


model.safetensors: reconstructing file:   0%|          |  0.00B / 1.21GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/314 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 33.4MB            

...

2_Dense/model.safetensors: downloading bytes:           |  0.00B            

config.json:   0%|          | 0.00/134 [00:00<?, ?B/s]

3_Dense/model.safetensors: reconstructing file:   0%|          |  0.00B / 9.44MB            

3_Dense/model.safetensors: downloading bytes:           |  0.00B            

document embeddings: (74, 768)
Q-01 What is form GR-305 used for?
  EmbeddingGemma
    1. GO-08  0.6955  Form GR-305 is used to appeal a decision of the Grant Office. It is th
    2. GO-12  0.4609  A student who disagrees with a decision may appeal within 14 calendar 
    3. GO-23  0.4505  Если студент не согласен с решением, он может подать апелляцию по форм


## My own queries

Write them, and their `relevant` chunk ids, **before** you run them. Say which one you expect BM25 to win and which you expect embeddings to win, and why.

| Id | Query | `relevant` | Expected winner, and why |
|---|---|---|---|
| Q-09 | What is form GR-210? | GO-07 | **BM25.** The rare token `210` appears in the query and in the chunk about that form. The embedding model sees three near-identical form names (GR-104, GR-210, GR-305) and may blur them. |
| Q-10 | Можно ли получать грант одновременно со стипендией? | GO-16 | **Embeddings.** The query is Russian and GO-16 is English, so they share no tokens. BM25 can only match the Russian chunks (GO-15, GO-23); a multilingual embedding model can still link the meaning. |

In [17]:
own_queries = [
    {"id": "Q-09", "text": "What is form GR-210?", "relevant": ["GO-07"]},
    {"id": "Q-10", "text": "Можно ли получать грант одновременно со стипендией?", "relevant": ["GO-16"]},
]
all_queries = fixed_queries + own_queries
print(len(all_queries), "queries")
for q in own_queries:
    print(q["id"], q["text"], q["relevant"])


10 queries
Q-09 What is form GR-210? ['GO-07']
Q-10 Можно ли получать грант одновременно со стипендией? ['GO-16']


## Results for all ten or more queries: top 3 from each retriever

In [18]:
table = []
for q in all_queries:
    b_scores = bm25.get_scores(tokenize(q["text"]))
    e_scores = emb_scores(q["text"])
    b_top = np.argsort(-b_scores)[:3]
    e_top = np.argsort(-e_scores)[:3]
    b_hit = any(ids[i] in q["relevant"] for i in b_top)
    e_hit = any(ids[i] in q["relevant"] for i in e_top)

    print(f"\n{q['id']}: {q['text']}    relevant = {q['relevant']}")
    show("BM25", b_top, b_scores)
    show("EmbeddingGemma", e_top, e_scores)
    print(f"  hit@3:  BM25 = {b_hit}   Embedding = {e_hit}")
    table.append((q["id"], [ids[i] for i in b_top], b_hit, [ids[i] for i in e_top], e_hit))

# summary in the format of the B3 table below
print("\n| Query | BM25 top 3 (ids) | BM25 hit@3 | Embedding top 3 (ids) | Embedding hit@3 |")
print("|---|---|:---:|---|:---:|")
for qid, bt, bh, et, eh in table:
    print(f"| {qid} | {', '.join(bt)} | {'✓' if bh else '✗'} | {', '.join(et)} | {'✓' if eh else '✗'} |")
print(f"| **hit@3 total** | | {sum(r[2] for r in table)}/{len(table)} | | {sum(r[4] for r in table)}/{len(table)} |")



Q-01: What is form GR-305 used for?    relevant = ['GO-08', 'GO-12', 'GO-23']
  BM25
    1. GO-08  15.8255  Form GR-305 is used to appeal a decision of the Grant Office. It is th
    2. GO-12  8.1953  A student who disagrees with a decision may appeal within 14 calendar 
    3. GO-06  8.0700  Form GR-104 is the application form for the need-based study grant. It
  EmbeddingGemma
    1. GO-08  0.6955  Form GR-305 is used to appeal a decision of the Grant Office. It is th
    2. GO-12  0.4609  A student who disagrees with a decision may appeal within 14 calendar 
    3. GO-23  0.4505  Если студент не согласен с решением, он может подать апелляцию по форм
  hit@3:  BM25 = True   Embedding = True

Q-02: My family is poor. Is there financial help for students like me?    relevant = ['GO-01', 'GO-02', 'GO-03']
  BM25
    1. KZ-17  8.7262  The lake currently covers about 16,400 km2 (6,300 sq mi). However, lik
    2. GO-20  8.3105  The need-based grant is for full-time undergraduate students 

## B3 — comparison

| Query | BM25 top 3 (ids) | BM25 hit@3 | Embedding top 3 (ids) | Embedding hit@3 |
|---|---|:---:|---|:---:|
| Q-01 | GO-08, GO-12, GO-06 | ✓ | GO-08, GO-12, GO-23 | ✓ |
| Q-02 | KZ-17, GO-20, KZ-25 | ✗ | GO-01, GO-10, GO-20 | ✓ |
| Q-03 | GO-14, GO-02, GO-01 | ✓ | GO-14, GO-09, GO-11 | ✓ |
| Q-04 | GO-05, GO-17, GO-19 | ✗ | GO-02, GO-05, GO-17 | ✓ |
| Q-05 | KZ-05, KZ-06, KZ-11 | ✓ | KZ-05, KZ-06, KZ-03 | ✓ |
| Q-06 | KZ-37, KZ-38, KZ-03 | ✓ | KZ-37, KZ-38, KZ-10 | ✓ |
| Q-07 | GO-21, GO-16, GO-01 | ✗ | GO-15, GO-18, GO-01 | ✓ |
| Q-08 | GO-17, GO-05, GO-04 | ✓ | GO-17, GO-18, GO-15 | ✓ |
| Q-09 | GO-07, GO-13, GO-10 | ✓ | GO-07, GO-10, GO-13 | ✓ |
| Q-10 | GO-23, GO-14, GO-15 | ✗ | GO-16, GO-15, GO-19 | ✓ |
| **hit@3 total** | | 6/10 | | 10/10 |

**Both retrievers agree:** which query, and why was it easy? Both retrievers agree: Q-06 (“Where did the Soviet Union test its nuclear weapons?”). Both returned KZ-37 and KZ-38 as the top two (BM25: 19.09 and 17.17; EmbeddingGemma: 0.5960 and 0.5102). It was easy because the content words of the query (Soviet Union, nuclear, test) appear in those chunks and the chunks say the same thing, so lexical match and meaning point to the same place. Q-05 (Astana/Nur-Sultan, KZ-05 first in both) is the same case.

>

**BM25 did better:** which query, and what exact string did it match that the embedding model blurred? BM25 did better:

No query: there is no case where BM25 had a relevant chunk in the top 3 and EmbeddingGemma did not (hit@3 6/10 vs 10/10). I expected Q-09 (form GR-210) to be the BM25 win, because 210 is a rare token, but that expectation was not confirmed: BM25 returned GO-07 first (10.62), and EmbeddingGemma also returned GO-07 first (0.6561), so it did not blur the form names. The only visible BM25 advantage is a wider score gap on exact-code queries, for example Q-01, where GR-305 gives GO-08 a score of 15.83 against 8.20 for the next chunk. That did not change hit@3.

>

**Embeddings did better:** which query, and what could BM25 not match? Embeddings did better:

Q-07 (“Can the grant be sent to my mother’s card?”). The relevant chunk GO-15 is in Russian ("на счёт… родителей"), so it shares no words with the English query. BM25 returned GO-21 (office room and hours), GO-16 and GO-01, which look like matches on common English words rather than on the topic (I did not check which tokens matched). EmbeddingGemma put GO-15 first because “card” and “mother” are close in meaning to “account” and “parents”. Q-02, Q-04 and Q-10 show the same pattern: the query and the chunk use different words (poor vs low-income; “minimum GPA” vs “GPA is at least 2.67”) or different languages.

>

---
# Part C — connect retrieval to the model (1.5 pt)

Model: `gpt-5.6-luna` for every call. Top 3 unless stated.

In [19]:
from openai import OpenAI
import tiktoken

client = OpenAI()                      # reads OPENAI_API_KEY from the environment (.env was loaded in Part B)
MODEL = "gpt-5.6-luna"
enc = tiktoken.get_encoding("o200k_base")

with open("data/qa.json", encoding="utf-8") as f:
    qa = json.load(f)["questions"]

SYSTEM = """Answer the question using only the provided context.
If the answer cannot be found in the context, say that you do not know.
Cite the context blocks you used, like [1] or [2][3]."""

def retrieve(question, method, k=3):
    s = bm25.get_scores(tokenize(question)) if method == "bm25" else emb_scores(question)
    order = np.argsort(-s)[:k]
    return order, s

def build_messages(question, order):
    blocks = "\n\n".join(f"[{n}] {texts[i]}" for n, i in enumerate(order, 1))
    user = f"Context:\n{blocks}\n\nQuestion: {question}"
    msgs = [{"role": "system", "content": SYSTEM},
            {"role": "user", "content": user}]
    return msgs, blocks

def ask(messages):
    r = client.chat.completions.create(model=MODEL, messages=messages)
    return r.choices[0].message.content, r.usage.prompt_tokens

print("model:", MODEL, "| questions:", [q["id"] for q in qa])

model: gpt-5.6-luna | questions: ['C-01', 'C-02', 'C-03', 'C-04', 'C-05', 'C-06']


## C1 — the complete prompt, printed before it is sent

In [21]:
%pip install -U brotli

  Attempting uninstall: brotli
    Found existing installation: Brotli 1.0.9
    Uninstalling Brotli-1.0.9:
      Successfully uninstalled Brotli-1.0.9
Note: you may need to restart the kernel to use updated packages.


In [23]:
q = qa[0]                                           # C-01
client = OpenAI(default_headers={"Accept-Encoding": "gzip, deflate"})
order, _ = retrieve(q["question"], "emb")
messages, blocks = build_messages(q["question"], order)

# the complete prompt, printed BEFORE it is sent
print("=== SYSTEM ===\n" + messages[0]["content"])
print("\n=== USER ===\n" + messages[1]["content"])

n_sys     = len(enc.encode(SYSTEM))
n_q       = len(enc.encode(q["question"]))
n_ctx     = len(enc.encode(blocks))
n_total   = sum(len(enc.encode(m["content"])) for m in messages)
per_chunk = [len(enc.encode(texts[i])) for i in order]

answer, usage_tokens = ask(messages)

print("\n--- token counts (tiktoken o200k_base) ---")
print("System instructions :", n_sys)
print("Question            :", n_q)
print("Retrieved context   :", n_ctx)
print("Total input (sum of message contents):", n_total)
print("usage.prompt_tokens from the response:", usage_tokens)
print("Number of retrieved chunks:", len(order))
print("Tokens per chunk:", per_chunk)
print("Total context size: tokens =", n_ctx, "| characters =", len(blocks))
print("\nANSWER:", answer)

=== SYSTEM ===
Answer the question using only the provided context.
If the answer cannot be found in the context, say that you do not know.
Cite the context blocks you used, like [1] or [2][3].

=== USER ===
Context:
[1] The grant amount depends only on the income band, not on the GPA. A student in band 1 receives 250,000 tenge for the academic year. A student in band 2 receives 150,000 tenge for the academic year. The amount is paid in two equal instalments, the first in October and the second in February. There is no partial grant: a student either receives the full amount for their band or nothing.

[2] The income band is set from the monthly household income per person, declared on form GR-210. Band 1: below 85,000 tenge per person per month. Band 2: from 85,000 up to 150,000 tenge per person per month. Band 3: above 150,000 tenge per person per month. Households in band 3 are not eligible for the need-based grant, although their students may still apply for merit scholarships, whi

| Part of the input | Tokens (tiktoken `o200k_base`) |
|---|---:|
| System instructions |44 |
| Question |28 |
| Retrieved context |288 |
| **Total input** |354 |
| **`usage.prompt_tokens` from the response** |364 |

*(Fill the numbers from the output of the cell above.)*

- Number of retrieved chunks: 3
- Tokens per chunk: [90, 104, 85]
- Total context size (tokens / characters): 1145

**Why do the tiktoken total and `usage.prompt_tokens` differ?**

> `usage.prompt_tokens` is larger than my tiktoken total. tiktoken counts only the text I give it. The API also adds tokens that I never see in my strings: special tokens that mark where each message starts and ends and which role it belongs to (system or user), and the framing of the whole conversation. My count also excludes the words "Context:" and "Question:" if I count the three parts separately, but they are part of the user message.

**What exactly does retrieval add to the model's input?**

> Only text. Retrieval adds the three retrieved chunks, copied word for word into the prompt and numbered, so they make up most of the context tokens. The model gets no index, no scores and no access to the corpus: it sees these strings and nothing else, plus the instructions and the question.

## C2 — three systems, six questions

In [24]:
res = {}
for q in qa:
    r = {"llm": ask([{"role": "user", "content": q["question"]}])[0]}
    for name in ("bm25", "emb"):
        order, scores = retrieve(q["question"], name)
        msgs, blocks = build_messages(q["question"], order)
        r[name] = {"ids": [ids[i] for i in order],
                   "scores": [float(scores[i]) for i in order],
                   "answer": ask(msgs)[0],
                   "blocks": blocks}
    res[q["id"]] = r

    print(f"\n{q['id']}: {q['question']}")
    print("REFERENCE :", q["reference"])
    print("LLM only  :", r["llm"])
    print("BM25 RAG  :", r["bm25"]["ids"], "->", r["bm25"]["answer"])
    print("Emb RAG   :", r["emb"]["ids"], "->", r["emb"]["answer"])


C-01: How much does a student in income band 2 receive, and when is it paid?
REFERENCE : 150,000 tenge for the academic year, in two equal instalments: October and February.
LLM only  : Which country or student-aid scheme do you mean? “Income band 2” is used by several programs, and the amount and payment date depend on the scheme and academic year.
BM25 RAG  : ['GO-04', 'GO-02', 'GO-01'] -> A student in income band 2 receives **150,000 tenge for the academic year**, paid in **two equal instalments**: **75,000 tenge in October** and **75,000 tenge in February**. [1]
Emb RAG   : ['GO-04', 'GO-03', 'GO-15'] -> A student in income band 2 receives **150,000 tenge per academic year**, paid in **two equal instalments of 75,000 tenge**: one in **October** and one in **February**. [1]

C-02: My grant application was refused. How long do I have to appeal, and which form do I use?
REFERENCE : 14 calendar days from the date on the decision letter, using form GR-305.
LLM only  : The deadline and 

Mark each answer ✓ (agrees with `reference`) or ✗, and give a few words of the answer.

| Question | LLM only | BM25 RAG | Embedding RAG |
|---|---|---|---|
| C-01 |✗ asks which country/scheme |✓ 150,000; Oct + Feb |✓ 150,000; Oct + Feb |
| C-02 |✗ asks jurisdiction | ✓ 14 days; GR-305| ✓ 14 days; GR-305|
| C-03 | ✗ asks which institution|✓ 2.67 | ✓ 2.67|
| C-04 | ✓ Oct 2017; 2025 → 2031|✓ Oct 2017; 2025 → 2031 |✓ Oct 2017; 2031 |
| C-05 | ✓ 1963; Alma-Ata Institute of National Economy| ✓ 1963; Almaty Institute of National Economy| ✓ 1963; Almaty Institute of National Economy|
| C-06 |✗ generic US answer | ✓ “I do not know”| ✓ no amount invented; separate scheme, ask department|
| **agrees with `reference`** | 2/6 | 6/6 | 6/6 |

## C3 — question 1 of 3: C-03

**1. What BM25 retrieved** (top 3, ids and scores):

> 1. GO-16 — 15.7985
> 2. GO-02 — 14.0587 (the relevant chunk)
> 3. GO-06 — 13.8516
>
> The three scores are close together, so BM25 hardly separates the relevant chunk from the other two.

**2. What EmbeddingGemma retrieved** (top 3, ids and scores):

> 1. GO-02 — 0.6286 (the relevant chunk)
> 2. GO-20 — 0.5691
> 3. GO-05 — 0.5451 (the archived 2025 rule)

**3. What was sent to the model** (the context blocks):

> **BM25 prompt:**
> [1] (GO-16) "The need-based grant can be held together with a merit scholarship, but not together with a second need-based grant…"
> [2] (GO-02) "An applicant qualifies for the 2026 grant when all three conditions hold… their cumulative GPA is at least 2.67 on the 4.0 scale…"
> [3] (GO-06) "Form GR-104 is the application form for the need-based study grant…"
>
> **EmbeddingGemma prompt:**
> [1] (GO-02) the same eligibility chunk with the 2.67 rule
> [2] (GO-20) "The need-based grant is for full-time undergraduate students only…"
> [3] (GO-05) "ARCHIVED — kept for reference only. Under the 2025 scheme the minimum GPA for the grant was 3.0… do not apply to any application made for the 2026 academic year."
>
> (The full text of every block is printed in the cell below.)

**4. How retrieval changed the answer:** more accurate, more specific, more grounded, or wrong because the wrong chunks came back?

> It made the answer accurate, specific and grounded. Without retrieval the model only asked which grant or institution I meant and gave no number. With retrieval both systems answered "The minimum GPA is 2.67 on a 4.0 scale" and cited the block that holds it ([2] for BM25, [1] for EmbeddingGemma). The answer matches the reference.

**5. What was missed, and was it a retrieval problem or a generation problem?** Quote the evidence.

> Nothing in the final answers was missed or wrong, so there was no failure. But retrieval was imperfect in both systems, and the model had to cope with it:
>
> - **BM25:** the relevant chunk came second (14.06), behind GO-16 (15.80), a chunk about combining grants that does not mention the GPA rule. With K = 1 the model would not have seen GO-02 at all. This would have been a retrieval problem.
> - **EmbeddingGemma:** the relevant chunk came first, but the archived GO-05 with the old 3.0 rule was third (0.5451) and went into the prompt. This is a retrieval problem (a stale chunk retrieved), not a generation problem, because the model handled it correctly: it answered 2.67, and the chunk itself says it is archived and "do[es] not apply to any application made for the 2026 academic year."
>
> So the only risk was in retrieval; generation resolved the conflict between 2.67 and 3.0 correctly.

In [25]:
def inspect_question(qid):
    q = next(x for x in qa if x["id"] == qid)
    r = res[qid]
    print(qid, "-", q["question"])
    print("REFERENCE:", q["reference"])
    print("relevant chunks:", q["relevant"])
    for name, label in (("bm25", "BM25"), ("emb", "EmbeddingGemma")):
        print(f"\n--- {label}: top 3 (id, score) ---")
        for cid, sc in zip(r[name]["ids"], r[name]["scores"]):
            print(f"  {cid}  {sc:.4f}")
        print(f"--- context sent to the model ({label}) ---")
        print(r[name]["blocks"])
        print(f"ANSWER ({label}):", r[name]["answer"])
    print("\nLLM only:", r["llm"])

inspect_question("C-03")

C-03 - What is the minimum GPA needed to qualify for the need-based grant?
REFERENCE: A cumulative GPA of at least 2.67 on the 4.0 scale. (The 3.0 in GO-05 is the archived 2025 rule.)
relevant chunks: ['GO-02']

--- BM25: top 3 (id, score) ---
  GO-16  15.7985
  GO-02  14.0587
  GO-06  13.8516
--- context sent to the model (BM25) ---
[1] The need-based grant can be held together with a merit scholarship, but not together with a second need-based grant from any other source. The total of all university support a student receives in one academic year is capped at 400,000 tenge. When the cap would be exceeded, the need-based grant is reduced so that the total equals the cap.

[2] An applicant qualifies for the 2026 grant when all three conditions hold at the time of the decision: their cumulative GPA is at least 2.67 on the 4.0 scale; their household is in income band 1 or income band 2; and both a transcript and a copy of the national id card are on file with the Grant Office. Meeting tw

## C3 — question 2 of 3: C-06

**1. What BM25 retrieved** (top 3, ids and scores):

> 1. GO-16 — 16.9958
> 2. GO-01 — 14.1725
> 3. GO-20 — 13.7202 (the relevant chunk)

**2. What EmbeddingGemma retrieved** (top 3, ids and scores):

> 1. GO-20 — 0.6664 (the relevant chunk)
> 2. GO-16 — 0.6161
> 3. GO-01 — 0.5539

**3. What was sent to the model** (the context blocks):

> **BM25 prompt:**
> [1] (GO-16) "The need-based grant can be held together with a merit scholarship, but not together with a second need-based grant…"
> [2] (GO-01) "The need-based study grant 2026 is run by the Grant Office for full-time undergraduate students…"
> [3] (GO-20) "The need-based grant is for full-time undergraduate students only… Master's and doctoral students are supported by a separate scheme, which this handbook does not describe…"
>
> **EmbeddingGemma prompt:** the same three chunks in the order GO-20, GO-16, GO-01.
>
> (The full text of every block is printed in the cell below.)

**4. How retrieval changed the answer:** more accurate, more specific, more grounded, or wrong because the wrong chunks came back?

> It made the answer grounded and honest about what is not known. Without retrieval the model gave a long generic answer that assumed the United States (Pell Grants, Federal Work-Study, loans) and invented a context that has nothing to do with this handbook. With retrieval, BM25 answered "I do not know how much you can get… master's students are supported by a separate scheme, and its rules are not described here", which is exactly the reference. EmbeddingGemma answered "You cannot apply for this need-based grant because it is only for full-time undergraduate students… ask their department", without inventing an amount.

**5. What was missed, and was it a retrieval problem or a generation problem?** Quote the evidence.

> Nothing was missed by retrieval: GO-20, the only chunk that speaks about master's students, was in both prompts (third for BM25, first for EmbeddingGemma). The corpus does not contain the amount, so the right answer is "I do not know", and the evidence is in GO-20 itself: "Master's and doctoral students are supported by a separate scheme, which this handbook does not describe".
>
> The difference between the two answers is in generation, not retrieval. BM25 RAG said it does not know, as the reference requires. EmbeddingGemma RAG did not use the words "I do not know" and stated more firmly "You cannot apply for this need-based grant". That follows from GO-20 ("for full-time undergraduate students only"), so it is grounded, but it is a stronger claim than the context supports about the master's scheme. It invented no number.

In [26]:
inspect_question("C-06")

C-06 - I am a master's student. How much need-based grant can I get?
REFERENCE: The corpus does not say. GO-20 says master's students have a separate scheme that the handbook does not describe.
relevant chunks: ['GO-20']

--- BM25: top 3 (id, score) ---
  GO-16  16.9958
  GO-01  14.1725
  GO-20  13.7202
--- context sent to the model (BM25) ---
[1] The need-based grant can be held together with a merit scholarship, but not together with a second need-based grant from any other source. The total of all university support a student receives in one academic year is capped at 400,000 tenge. When the cap would be exceeded, the need-based grant is reduced so that the total equals the cap.

[2] The need-based study grant 2026 is run by the Grant Office for full-time undergraduate students. It exists so that a student from a low-income household can pay for study costs without taking on paid work that harms their studies. One grant is awarded per student per academic year. The grant is a gift, 

## C3 — question 3 of 3: C-02

**1. What BM25 retrieved** (top 3, ids and scores):

> 1. GO-05 — 17.2414 (the archived 2025 rule, not relevant)
> 2. GO-08 — 14.5287 (relevant)
> 3. GO-12 — 13.5735 (relevant)

**2. What EmbeddingGemma retrieved** (top 3, ids and scores):

> 1. GO-08 — 0.6029 (relevant)
> 2. GO-12 — 0.5591 (relevant)
> 3. GO-11 — 0.5343 (not relevant)

**3. What was sent to the model** (the context blocks):

> **BM25 prompt:**
> [1] (GO-05) "ARCHIVED — kept for reference only. Under the 2025 scheme the minimum GPA for the grant was 3.0…"
> [2] (GO-08) "Form GR-305 is used to appeal a decision of the Grant Office. It is the appeal form and nothing else…"
> [3] (GO-12) "A student who disagrees with a decision may appeal within 14 calendar days of the date on the decision letter, using form GR-305…"
>
> **EmbeddingGemma prompt:**
> [1] (GO-08) the GR-305 appeal form chunk
> [2] (GO-12) the 14-day appeal chunk
> [3] (GO-11) "If a required document is missing, the Grant Office does not refuse the application straight away… The student then has 10 working days to upload it…"
>
> (The full text of every block is printed in the cell below.)

**4. How retrieval changed the answer:** more accurate, more specific, more grounded, or wrong because the wrong chunks came back?

> It made the answer accurate and specific. Without retrieval the model could not answer: it said the deadline and the form depend on the country and the agency, and asked for my jurisdiction and the grant name. With retrieval both systems answered "You have 14 calendar days from the date on the decision letter to appeal. Use form GR-305", which matches the reference. The citations ([2][3] for BM25, [1][2] for EmbeddingGemma) point to the two relevant chunks.

**5. What was missed, and was it a retrieval problem or a generation problem?** Quote the evidence.

> Nothing was missed in the final answers, but both retrievers put a distracting chunk in the prompt, so the retrieval was noisy while generation was robust.
>
> - **BM25** ranked the archived GO-05 first (17.24), above both relevant chunks. It is about the 2025 GPA rule and has nothing to do with appeals. I did not check which tokens matched.
> - **EmbeddingGemma** put GO-11 third (0.5343). It is also off-topic, and it contains a different deadline ("10 working days"), which could have been confused with the 14 days.
> - Neither retriever returned GO-23, the Russian chunk about appeals, so only 2 of the 3 relevant chunks were found by each. This did no harm, because the answer is fully in GO-08 and GO-12.
>
> The failure risk was in retrieval (irrelevant chunks), but the model ignored it: its citations are the relevant chunks only, and it did not use the 3.0 rule or the 10-day deadline. So this was not a generation problem.

In [27]:
inspect_question("C-02")

C-02 - My grant application was refused. How long do I have to appeal, and which form do I use?
REFERENCE: 14 calendar days from the date on the decision letter, using form GR-305.
relevant chunks: ['GO-08', 'GO-12', 'GO-23']

--- BM25: top 3 (id, score) ---
  GO-05  17.2414
  GO-08  14.5287
  GO-12  13.5735
--- context sent to the model (BM25) ---
[1] ARCHIVED — kept for reference only. Under the 2025 scheme the minimum GPA for the grant was 3.0, band 1 received 200,000 tenge, band 2 received 120,000 tenge, and the whole amount was paid in a single instalment in November. These rules were replaced on 1 June 2026 and do not apply to any application made for the 2026 academic year.

[2] Form GR-305 is used to appeal a decision of the Grant Office. It is the appeal form and nothing else: it cannot be used to apply for the first time or to report a change in circumstances. On GR-305 the student states which part of the decision they believe is wrong and which rule in this handbook support

---
# Final questions

**1. What is the main difference between BM25 and embedding retrieval?**

> BM25 is lexical: it scores a chunk by the query words it contains, weighted by how rare each word is in the corpus and corrected for chunk length. An embedding model turns the query and every chunk into vectors and ranks by how close they are in meaning. BM25 needs the same words; embeddings do not.

**2. When can BM25 perform better than embeddings?**

> When the answer depends on an exact, rare string: a form code such as GR-305, an id, a number, a name, an unusual term. BM25 gives such a token a high weight, while an embedding model may blur similar-looking strings (GR-104, GR-210, GR-305) into one neighbourhood.

**3. When can embeddings perform better than BM25?**

> When the query and the chunk say the same thing in different words: synonyms, paraphrases, abbreviations (RAG vs "Retrieval-Augmented Generation") and different languages (a Russian or Kazakh query over an English chunk). BM25 finds no shared token and scores zero.

**4. Why is cosine similarity used with embeddings?**

> Cosine compares the direction of two vectors and ignores their length, and in an embedding space direction carries the meaning. It gives a score between -1 and 1 that is comparable across chunks, so the chunks can be ranked.

**5. What information is actually sent to an LLM in a RAG system?**

> A single prompt: the system instructions, the text of the top-K retrieved chunks, and the user's question. Nothing else from the corpus is sent.

**6. Does the LLM search the original documents itself?**

> No. Retrieval is ordinary code that runs before the model call. The model sees only the text in its prompt and cannot open, search or scroll the documents.

**7. What happens if retrieval returns an irrelevant chunk?**

> The model is given wrong or distracting material. It may answer from it as if it were true (for example the archived 2025 rule instead of the current one), or ignore it and answer from its own memory. Generation cannot repair a retrieval failure: if the right chunk is not in the prompt, the model cannot use it. The irrelevant chunk also uses tokens.

**8. Why can increasing Top-K sometimes improve RAG and sometimes make it worse?** Report K = 1, 3 and 8 on one question. With K = 1 the prompt had 177 tokens and contained only GO-02; the answer was correct. With K = 3 it had 326 tokens and added GO-20 and the archived GO-05; still correct. With K = 8 it had 757 tokens and added GO-16, GO-01, GO-06, GO-19 and GO-17; still correct. A larger K makes it more likely that the right chunk is included (with BM25 on C-03 the right chunk was only second, so K = 1 would have failed), but it adds irrelevant and sometimes contradictory text such as the 3.0 rule in GO-05, and the cost grows (757 vs 177 tokens). Here the embedding retriever already had GO-02 first, so extra chunks added cost without improving the answer.

>

In [28]:
q = next(x for x in qa if x["id"] == "C-03")
for k in (1, 3, 8):
    order, _ = retrieve(q["question"], "emb", k=k)
    msgs, _ = build_messages(q["question"], order)
    ans, tokens = ask(msgs)
    print(f"K = {k}  | prompt_tokens = {tokens}  | chunks = {[ids[i] for i in order]}")
    print("ANSWER:", ans, "\n")

K = 1  | prompt_tokens = 177  | chunks = ['GO-02']
ANSWER: The minimum GPA is **2.67 on a 4.0 scale**. [1] 

K = 3  | prompt_tokens = 326  | chunks = ['GO-02', 'GO-20', 'GO-05']
ANSWER: The minimum GPA needed is **2.67 on a 4.0 scale**. [1] 

K = 8  | prompt_tokens = 757  | chunks = ['GO-02', 'GO-20', 'GO-05', 'GO-16', 'GO-01', 'GO-06', 'GO-19', 'GO-17']
ANSWER: The minimum GPA is **2.67 on a 4.0 scale**. [1] 



**9. Is a vector database required to build a RAG system?**

> No. In this homework the embeddings sit in a plain NumPy array (`doc_emb`) and the similarity is one matrix operation, which is enough for 74 chunks. A vector database helps when the corpus is large (many thousands to millions of chunks) and needs fast approximate search, persistence, filtering or updates.

**10. Which retrieval approach worked better for this corpus, and what evidence supports that?** Give hit@3 for both over your ten queries and the C2 agreement counts for both RAG systems.
EmbeddingGemma worked better at retrieval: hit@3 was 10/10 for EmbeddingGemma against 6/10 for BM25 over my ten queries. All four BM25 misses were vocabulary or language mismatches: Q-02 (“poor” vs “low-income”), Q-04 (“minimum GPA” matched the archived GO-05 first and the right chunk GO-02 did not make the top 3), Q-07 (English query, Russian chunk GO-15) and Q-10 (Russian query, English chunk GO-16).

End to end the difference disappeared on the six C2 questions: both RAG systems agreed with the reference on 6/6, while the LLM alone agreed on 2/6 (only the two Wikipedia questions C-04 and C-05). BM25 still succeeded there because the six questions share words with the right chunks, and because top 3 gave it a second chance: in C-03 the right chunk GO-02 was second, in C-06 GO-20 was third. So the evidence supports embeddings as the more reliable retriever on this corpus, with the caveat that ten and six questions are small samples and both retrievers brought the archived GO-05 into the context at least once without harm to the answer.

>

---
# Conclusion
RAG changed the model’s answers on the handbook questions from “which country do you mean?” (C-01 to C-03) or a generic US answer (C-06) to the correct, specific answer in 6 of 6 cases, because retrieval put the right text into the prompt; for the Wikipedia questions (C-04, C-05) the model already knew the facts and RAG added nothing. The model never searches the documents; it only sees the retrieved chunks, so every error has to be located either in retrieval (the right chunk missing from the top 3) or in generation (the right chunk present, answer wrong). In this experiment I found no generation failures: where retrieval put a distracting chunk in the context (the archived GO-05 in C-02 and C-03), the model still answered correctly. BM25 is cheap and strong on exact strings but fails when the query uses other words or another language; embeddings handle paraphrase and cross-language queries, at the cost of loading a 1.2 GB model. Top-K is a trade-off between recall and cost and noise: on C-03 the prompt grew from 177 tokens at K = 1 to 757 at K = 8 with no gain.

>